# Boosting Assignment

This notebook covers theory and practical questions on Boosting techniques.


## Setup

If `xgboost` or `catboost` are missing, run the install cell below once.


In [ ]:
# Optional: install additional libs
# !pip -q install xgboost catboost

## Theory Answers

**1. What is Boosting in Machine Learning?**
Boosting is an ensemble technique that builds models sequentially, where each new model focuses on correcting the errors of the previous ones. The final prediction is a weighted combination of all models.

**2. How does Boosting differ from Bagging?**
Bagging trains models independently on bootstrapped data and averages them to reduce variance. Boosting trains models sequentially, reweighting data to reduce bias and improve difficult cases.

**3. What is the key idea behind AdaBoost?**
AdaBoost assigns higher weights to misclassified samples so subsequent weak learners focus more on them. It combines weak learners with weighted votes.

**4. Explain the working of AdaBoost with an example.**
Example: For binary classification, start with equal weights. Train a weak classifier (e.g., decision stump). Misclassified points get higher weights. Train another stump with new weights. Final prediction is a weighted vote of stumps.

**5. What is Gradient Boosting, and how is it different from AdaBoost?**
Gradient Boosting fits new models to the negative gradient of a loss function, enabling optimization of arbitrary differentiable losses. AdaBoost is a specific boosting algorithm that reweights samples based on classification errors.

**6. What is the loss function in Gradient Boosting?**
It depends on the task: squared error for regression, log loss for classification, etc. The algorithm minimizes the chosen loss via gradient descent in function space.

**7. How does XGBoost improve over traditional Gradient Boosting?**
XGBoost adds regularization, efficient handling of sparse data, parallelization, and optimized tree building. It also includes shrinkage and column subsampling.

**8. What is the difference between XGBoost and CatBoost?**
XGBoost is optimized gradient boosting with advanced regularization. CatBoost is designed to handle categorical features efficiently with ordered boosting and target encoding.

**9. What are some real-world applications of Boosting techniques?**
Credit scoring, fraud detection, ranking systems, medical diagnosis, churn prediction, and demand forecasting.

**10. How does regularization help in XGBoost?**
It penalizes model complexity (tree depth, leaf weights), reducing overfitting and improving generalization.

**11. What are some hyperparameters to tune in Gradient Boosting models?**
Number of estimators, learning rate, max depth, min samples split/leaf, subsample, and max features.

**12. What is the concept of Feature Importance in Boosting?**
Feature importance measures how much each feature contributes to improving splits or reducing loss across the ensemble.

**13. Why is CatBoost efficient for categorical data?**
It uses ordered target statistics and built-in categorical handling, reducing leakage and preprocessing overhead.


## Practical Tasks (Start from 14)

We will use scikit-learn datasets and metrics. Each task is in its own cell.

**Practical numbering starts from 14 onward.**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score, f1_score, mean_squared_error, roc_curve, auc, confusion_matrix, log_loss
from sklearn.datasets import load_breast_cancer, load_diabetes, make_classification, load_iris
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

sns.set(style="whitegrid")

In [ ]:
# 14. AdaBoost Classifier on a sample dataset
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

ada_clf = AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=42)
ada_clf.fit(X_train, y_train)

y_pred = ada_clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
# 15. AdaBoost Regressor with MAE
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

ada_reg = AdaBoostRegressor(n_estimators=200, learning_rate=0.5, random_state=42)
ada_reg.fit(X_train, y_train)

y_pred = ada_reg.predict(X_test)
print("MAE:", mean_absolute_error(y_test, y_pred))

In [ ]:
# 16. Gradient Boosting Classifier feature importance
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(X_train, y_train)

importances = gb_clf.feature_importances_
print("Top 10 importances:")
print(np.sort(importances)[-10:])

In [ ]:
# 17. Gradient Boosting Regressor with R2
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gb_reg = GradientBoostingRegressor(random_state=42)
gb_reg.fit(X_train, y_train)

y_pred = gb_reg.predict(X_test)
print("R2:", r2_score(y_test, y_pred))

In [ ]:
# 18. XGBoost vs Gradient Boosting
try:
    from xgboost import XGBClassifier
    X, y = load_breast_cancer(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    xgb = XGBClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss"
    )
    xgb.fit(X_train, y_train)
    xgb_pred = xgb.predict(X_test)

    gb_clf = GradientBoostingClassifier(random_state=42)
    gb_clf.fit(X_train, y_train)
    gb_pred = gb_clf.predict(X_test)

    print("XGBoost Accuracy:", accuracy_score(y_test, xgb_pred))
    print("Gradient Boosting Accuracy:", accuracy_score(y_test, gb_pred))
except Exception as e:
    print("XGBoost not available:", e)

In [ ]:
# 19. CatBoost Classifier with F1-score
try:
    from catboost import CatBoostClassifier
    X, y = load_breast_cancer(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    cat = CatBoostClassifier(iterations=300, learning_rate=0.1, depth=6, verbose=False, random_state=42)
    cat.fit(X_train, y_train)
    y_pred = cat.predict(X_test)
    print("F1-score:", f1_score(y_test, y_pred))
except Exception as e:
    print("CatBoost not available:", e)

In [ ]:
# 20. XGBoost Regressor with MSE
try:
    from xgboost import XGBRegressor
    X, y = load_diabetes(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    xgb_reg = XGBRegressor(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    xgb_reg.fit(X_train, y_train)
    y_pred = xgb_reg.predict(X_test)
    print("MSE:", mean_squared_error(y_test, y_pred))
except Exception as e:
    print("XGBoost not available:", e)

In [ ]:
# 21. AdaBoost feature importance
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

ada_clf = AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=42)
ada_clf.fit(X_train, y_train)

plt.figure(figsize=(8,4))
plt.plot(ada_clf.feature_importances_)
plt.title("AdaBoost Feature Importance")
plt.xlabel("Feature Index")
plt.ylabel("Importance")
plt.show()

In [ ]:
# 22. Gradient Boosting Regressor learning curves
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gb_reg = GradientBoostingRegressor(random_state=42)

test_scores = []
train_scores = []
for n_estimators in range(50, 401, 50):
    gb_reg.set_params(n_estimators=n_estimators)
    gb_reg.fit(X_train, y_train)
    train_scores.append(gb_reg.score(X_train, y_train))
    test_scores.append(gb_reg.score(X_test, y_test))

plt.figure(figsize=(8,4))
plt.plot(range(50, 401, 50), train_scores, label="Train R2")
plt.plot(range(50, 401, 50), test_scores, label="Test R2")
plt.xlabel("n_estimators")
plt.ylabel("R2")
plt.title("Gradient Boosting Learning Curve")
plt.legend()
plt.show()

In [ ]:
# 23. XGBoost feature importance
try:
    from xgboost import XGBClassifier, plot_importance
    X, y = load_breast_cancer(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    xgb = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=3, random_state=42, eval_metric="logloss")
    xgb.fit(X_train, y_train)

    plt.figure(figsize=(8,4))
    plot_importance(xgb, max_num_features=10)
    plt.title("XGBoost Feature Importance")
    plt.show()
except Exception as e:
    print("XGBoost not available:", e)

In [ ]:
# 24. CatBoost confusion matrix
try:
    from catboost import CatBoostClassifier
    X, y = load_breast_cancer(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    cat = CatBoostClassifier(iterations=300, learning_rate=0.1, depth=6, verbose=False, random_state=42)
    cat.fit(X_train, y_train)
    y_pred = cat.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("CatBoost Confusion Matrix")
    plt.ylabel("True")
    plt.xlabel("Pred")
    plt.show()
except Exception as e:
    print("CatBoost not available:", e)

In [ ]:
# 25. AdaBoost with different n_estimators
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

for n in [50, 100, 200, 400]:
    ada = AdaBoostClassifier(n_estimators=n, learning_rate=0.5, random_state=42)
    ada.fit(X_train, y_train)
    pred = ada.predict(X_test)
    print(f"n_estimators={n} -> accuracy={accuracy_score(y_test, pred):.4f}")

In [ ]:
# 26. Gradient Boosting ROC curve
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(X_train, y_train)

y_proba = gb_clf.predict_proba(X_test)[:,1]

fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(5,4))
plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Gradient Boosting ROC")
plt.legend()
plt.show()

In [ ]:
# 27. XGBoost Regressor tuning learning_rate with GridSearchCV
try:
    from xgboost import XGBRegressor
    X, y = load_diabetes(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    param_grid = {"learning_rate": [0.01, 0.05, 0.1, 0.2]}
    xgb = XGBRegressor(n_estimators=300, max_depth=3, subsample=0.8, colsample_bytree=0.8, random_state=42)

    grid = GridSearchCV(xgb, param_grid=param_grid, scoring="neg_mean_squared_error", cv=3)
    grid.fit(X_train, y_train)

    print("Best learning_rate:", grid.best_params_)
    print("Best CV MSE:", -grid.best_score_)
except Exception as e:
    print("XGBoost not available:", e)

In [ ]:
# 28. CatBoost on imbalanced dataset with class weights
try:
    from catboost import CatBoostClassifier
    X, y = make_classification(n_samples=2000, n_features=20, n_informative=5,
                               n_redundant=2, weights=[0.9, 0.1], random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # class weights: inverse of frequency
    class_weights = [1.0, 9.0]
    cat = CatBoostClassifier(iterations=300, learning_rate=0.1, depth=6,
                             class_weights=class_weights, verbose=False, random_state=42)
    cat.fit(X_train, y_train)
    y_pred = cat.predict(X_test)
    print("F1-score:", f1_score(y_test, y_pred))
except Exception as e:
    print("CatBoost not available:", e)

In [ ]:
# 29. AdaBoost learning rate effect
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

for lr in [0.01, 0.1, 0.5, 1.0]:
    ada = AdaBoostClassifier(n_estimators=200, learning_rate=lr, random_state=42)
    ada.fit(X_train, y_train)
    pred = ada.predict(X_test)
    print(f"learning_rate={lr} -> accuracy={accuracy_score(y_test, pred):.4f}")

In [ ]:
# 30. XGBoost multi-class classification with log-loss
try:
    from xgboost import XGBClassifier
    X, y = load_iris(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    xgb = XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="mlogloss"
    )
    xgb.fit(X_train, y_train)
    proba = xgb.predict_proba(X_test)
    print("Log-loss:", log_loss(y_test, proba))
except Exception as e:
    print("XGBoost not available:", e)